# Laboratorio guiado de exploración: color, paleta y segmentación

Este cuaderno está pensado para una exposición del ayudante de trabajos prácticos. La idea no es presentar una versión limpia y cerrada de un procedimiento, sino modelar una forma de explorar: mirar una imagen, formular hipótesis, probar cambios y justificar decisiones.


## Objetivo

Explorar cómo cambian la lectura y el procesamiento de una imagen cuando modificamos muestreo, conversión a grises, cuantización, paleta y criterios de segmentación.

## Resultados de aprendizaje

Al finalizar este laboratorio, vas a poder:

- comparar dos maneras de muestrear una misma imagen;
- analizar diferencias entre conversiones a escala de grises;
- relacionar cuantización y paleta de colores;
- formular y ajustar un criterio de segmentación por canal;
- limpiar una máscara y justificar por qué una estrategia resulta más útil que otra.

## Relación con la secuencia

Este cuaderno no reemplaza a los notebooks centrales de la unidad. Funciona como una capa de exploración guiada. La idea es recuperar algo valioso de los integradores de 2025: comparar métodos, detectar dificultades reales y discutir qué decisión conviene tomar cuando una técnica no funciona de manera inmediata.


## Cómo conviene usar este laboratorio en clase

- El ayudante puede detenerse en cada sección y pedir una predicción antes de ejecutar el bloque.
- Conviene escribir en el pizarrón o decir en voz alta una hipótesis breve antes de cambiar parámetros.
- Si una prueba falla, no hay que ocultarla: justamente esa falla forma parte del aprendizaje.
- El objetivo no es llegar rápido al resultado final, sino hacer visible el criterio técnico que organiza las decisiones.


## Módulos que vamos a usar

- `pathlib.Path`: para trabajar con rutas de archivo de manera clara.
- `numpy`: para operar con matrices y construir máscaras.
- `cv2`: para leer imágenes, convertir espacios de color y hacer operaciones morfológicas.
- `matplotlib.pyplot`: para visualizar comparaciones.
- `PIL.Image`: para abrir imágenes con Pillow y comparar comportamientos con OpenCV.


In [ ]:
# Importamos herramientas de rutas para ubicar la imagen de trabajo.
from pathlib import Path

# Importamos NumPy para operar con arreglos y máscaras.
import numpy as np

# Importamos OpenCV para leer imágenes y aplicar transformaciones.
import cv2

# Importamos Matplotlib para visualizar resultados dentro del cuaderno.
import matplotlib.pyplot as plt

# Importamos Pillow para comparar su comportamiento con OpenCV.
from PIL import Image

# Definimos la ruta de la imagen base para este laboratorio.
ruta_imagen = Path("Imagenes") / "globos.jpg"

print(f"Imagen seleccionada: {ruta_imagen}")


## 1. Imagen base y primera pregunta

Vamos a trabajar siempre sobre la misma imagen para que cualquier diferencia se deba al método y no a un cambio de escena.

**Pregunta para abrir la discusión:**

Si tuvieras que segmentar un color de esta imagen, ¿cuál te parece que sería más fácil de aislar? ¿Rojo, naranja, verde o azul?


In [ ]:
# Abrimos la imagen con Pillow.
imagen_pillow = Image.open(ruta_imagen)

# Abrimos la misma imagen con OpenCV.
imagen_opencv_bgr = cv2.imread(str(ruta_imagen), cv2.IMREAD_COLOR)
if imagen_opencv_bgr is None:
    raise FileNotFoundError(f"No se pudo leer la imagen: {ruta_imagen}")

# Convertimos la imagen de OpenCV a RGB para verla correctamente con Matplotlib.
imagen_opencv_rgb = cv2.cvtColor(imagen_opencv_bgr, cv2.COLOR_BGR2RGB)

print(f"Modo en Pillow: {imagen_pillow.mode}")
print(f"Tamaño en Pillow: {imagen_pillow.size}")
print(f"Forma en OpenCV: {imagen_opencv_bgr.shape}")

plt.figure(figsize=(10, 6), constrained_layout=True)
plt.imshow(imagen_opencv_rgb)
plt.title("Imagen base para el laboratorio", fontweight="bold", loc="left")
plt.axis("off")
plt.show()


Fijate que ya aparece una diferencia importante: Pillow y OpenCV no describen la imagen del mismo modo. Esa diferencia no siempre afecta el resultado final, pero sí importa cuando queremos razonar qué estructura tiene el dato con el que estamos trabajando.


## 2. Muestreo: dos formas de reducir la imagen

Ahora vamos a reducir la imagen de dos maneras distintas. Una usa `resize()` con Pillow y la otra toma un paso fijo sobre filas y columnas con OpenCV.

**Pregunta para el grupo:**

¿Esperan que ambas imágenes reducidas queden exactamente iguales o no? ¿Por qué?


In [ ]:
# Definimos un factor de muestreo editable para la exposición.
factor_muestreo = 4

# Muestreamos con Pillow usando resize.
imagen_muestreada_pillow = imagen_pillow.resize(
    (imagen_pillow.width // factor_muestreo, imagen_pillow.height // factor_muestreo)
)

# Muestreamos con OpenCV tomando una fila y una columna cada cierto paso.
imagen_muestreada_opencv = imagen_opencv_bgr[::factor_muestreo, ::factor_muestreo]
imagen_muestreada_opencv_rgb = cv2.cvtColor(imagen_muestreada_opencv, cv2.COLOR_BGR2RGB)

# Calculamos un porcentaje simple de reducción de cantidad de píxeles.
alto_original, ancho_original = imagen_opencv_bgr.shape[:2]
alto_muestreado, ancho_muestreado = imagen_muestreada_opencv.shape[:2]
pixeles_originales = alto_original * ancho_original
pixeles_muestreados = alto_muestreado * ancho_muestreado
porcentaje_reduccion = 100 * (1 - pixeles_muestreados / pixeles_originales)

print(f"Factor de muestreo: {factor_muestreo}")
print(f"Pixeles originales: {pixeles_originales}")
print(f"Pixeles tras el muestreo: {pixeles_muestreados}")
print(f"Reducción aproximada: {porcentaje_reduccion:.2f}%")


In [ ]:
# Comparamos ambos resultados de muestreo en una misma figura.
fig, ejes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

ejes[0].imshow(imagen_opencv_rgb)
ejes[0].set_title("Imagen original", fontweight="bold", loc="left")
ejes[0].axis("off")

ejes[1].imshow(imagen_muestreada_pillow)
ejes[1].set_title("Muestreo con Pillow", fontweight="bold", loc="left")
ejes[1].axis("off")

ejes[2].imshow(imagen_muestreada_opencv_rgb)
ejes[2].set_title("Muestreo con OpenCV", fontweight="bold", loc="left")
ejes[2].axis("off")

plt.show()


Las dos imágenes reducidas conservan la escena, pero no son idénticas. Eso ya habilita una discusión útil: reducir tamaño no es una operación neutra. También importa cómo se calcula esa reducción.


## 3. Escala de grises: tres formas de llegar a una imagen sin color

Vamos a comparar tres versiones:

1. la conversión que hace Pillow;
2. la conversión que hace OpenCV;
3. un promedio simple `1/3` por canal.

**Pregunta para abrir la comparación:**

Si las tres eliminan el color, ¿por qué podrían quedar distintas?


In [ ]:
# Convertimos la imagen muestreada con Pillow a escala de grises.
imagen_gris_pillow = imagen_muestreada_pillow.convert("L")

# Convertimos la imagen muestreada con OpenCV a escala de grises.
imagen_gris_opencv = cv2.cvtColor(imagen_muestreada_opencv, cv2.COLOR_BGR2GRAY)

# Calculamos una versión en grises promediando los tres canales por igual.
canales_opencv = imagen_muestreada_opencv.astype(np.float32)
imagen_gris_promedio = (canales_opencv[:, :, 0] + canales_opencv[:, :, 1] + canales_opencv[:, :, 2]) / 3
imagen_gris_promedio = imagen_gris_promedio.astype(np.uint8)

# Armamos una tabla simple de estadísticas para acompañar la lectura visual.
datos_grises = [
    ["Pillow", np.min(imagen_gris_pillow), np.max(imagen_gris_pillow), float(np.mean(imagen_gris_pillow))],
    ["OpenCV", np.min(imagen_gris_opencv), np.max(imagen_gris_opencv), float(np.mean(imagen_gris_opencv))],
    ["Promedio 1/3", np.min(imagen_gris_promedio), np.max(imagen_gris_promedio), float(np.mean(imagen_gris_promedio))],
]


In [ ]:
# Mostramos las tres conversiones y una tabla resumen.
fig, ejes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

ejes[0, 0].imshow(imagen_gris_pillow, cmap="gray")
ejes[0, 0].set_title("Grises con Pillow", fontweight="bold", loc="left")
ejes[0, 0].axis("off")

ejes[0, 1].imshow(imagen_gris_opencv, cmap="gray")
ejes[0, 1].set_title("Grises con OpenCV", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[1, 0].imshow(imagen_gris_promedio, cmap="gray")
ejes[1, 0].set_title("Promedio simple 1/3", fontweight="bold", loc="left")
ejes[1, 0].axis("off")

ejes[1, 1].axis("off")
tabla = ejes[1, 1].table(
    cellText=[[fila[0], int(fila[1]), int(fila[2]), f"{fila[3]:.2f}"] for fila in datos_grises],
    colLabels=["Método", "Mínimo", "Máximo", "Promedio"],
    loc="center",
    cellLoc="center",
    colLoc="center",
)
tabla.auto_set_font_size(False)
tabla.set_fontsize(10)
tabla.scale(1.2, 1.6)
ejes[1, 1].set_title("Comparación estadística", fontweight="bold", loc="left")

plt.show()


Acá aparece una idea importante: sacar color no siempre significa aplicar la misma fórmula. Mirá la imagen, pero también mirá la tabla. Si cambia el promedio, cambió la distribución general de intensidades, aunque a simple vista las tres versiones parezcan parecidas.


## 4. Cuantización y paleta de colores

En esta sección vamos a reducir la cantidad de colores disponibles. La idea es ver qué se gana en simplicidad y qué se pierde en detalle.

**Pregunta para el grupo:**

Si una imagen pasa de muchos colores a solo 4 o 8, ¿qué tipo de información se pierde primero?


In [ ]:
# Definimos una función para convertir una imagen a paleta adaptativa.
def cuantizar_con_paleta(imagen_pillow_rgb, cantidad_colores):
    imagen_cuantizada = imagen_pillow_rgb.convert("RGB").convert(
        "P",
        palette=Image.ADAPTIVE,
        colors=cantidad_colores,
    )
    paleta = imagen_cuantizada.getpalette()[: cantidad_colores * 3]
    colores = np.array(paleta, dtype=np.uint8).reshape(-1, 3)
    return imagen_cuantizada.convert("RGB"), colores

# Definimos una función para visualizar una paleta en forma de grilla.
def construir_grilla_paleta(colores, columnas=8, tamano_celda=24):
    cantidad = len(colores)
    filas = int(np.ceil(cantidad / columnas))
    grilla = np.full((filas * tamano_celda, columnas * tamano_celda, 3), 255, dtype=np.uint8)

    for indice, color in enumerate(colores):
        fila = indice // columnas
        columna = indice % columnas
        fila_inicio = fila * tamano_celda
        columna_inicio = columna * tamano_celda
        grilla[fila_inicio:fila_inicio + tamano_celda, columna_inicio:columna_inicio + tamano_celda] = color

    return grilla


In [ ]:
# Cuantizamos la imagen muestreada con tres niveles distintos.
imagen_cuantizada_64, paleta_64 = cuantizar_con_paleta(imagen_muestreada_pillow, 64)
imagen_cuantizada_16, paleta_16 = cuantizar_con_paleta(imagen_muestreada_pillow, 16)
imagen_cuantizada_4, paleta_4 = cuantizar_con_paleta(imagen_muestreada_pillow, 4)

# Construimos grillas visuales para cada paleta.
grilla_64 = construir_grilla_paleta(paleta_64, columnas=8)
grilla_16 = construir_grilla_paleta(paleta_16, columnas=8)
grilla_4 = construir_grilla_paleta(paleta_4, columnas=4)


In [ ]:
# Mostramos cada imagen cuantizada junto con su paleta.
fig, ejes = plt.subplots(3, 2, figsize=(14, 12), constrained_layout=True)

ejes[0, 0].imshow(imagen_cuantizada_64)
ejes[0, 0].set_title("Cuantización a 64 colores", fontweight="bold", loc="left")
ejes[0, 0].axis("off")
ejes[0, 1].imshow(grilla_64)
ejes[0, 1].set_title("Paleta de 64 colores", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[1, 0].imshow(imagen_cuantizada_16)
ejes[1, 0].set_title("Cuantización a 16 colores", fontweight="bold", loc="left")
ejes[1, 0].axis("off")
ejes[1, 1].imshow(grilla_16)
ejes[1, 1].set_title("Paleta de 16 colores", fontweight="bold", loc="left")
ejes[1, 1].axis("off")

ejes[2, 0].imshow(imagen_cuantizada_4)
ejes[2, 0].set_title("Cuantización a 4 colores", fontweight="bold", loc="left")
ejes[2, 0].axis("off")
ejes[2, 1].imshow(grilla_4)
ejes[2, 1].set_title("Paleta de 4 colores", fontweight="bold", loc="left")
ejes[2, 1].axis("off")

plt.show()


Fijate cómo cambia la relación entre la escena y la paleta. Cuando reducís mucho los colores, la imagen se vuelve más fácil de describir, pero también se pierden transiciones, sombras y matices. Acá aparece una tensión útil entre simplificación y fidelidad.


## 5. Qué dicen los canales antes de segmentar

Antes de elegir umbrales, conviene mirar los canales por separado. Eso ayuda a formular una hipótesis mejor sobre qué combinación puede aislar una región de interés.

En este laboratorio vamos a probar con los globos de color naranja.


In [ ]:
# Separamos la imagen original en canales RGB.
canal_rojo = imagen_opencv_rgb[:, :, 0]
canal_verde = imagen_opencv_rgb[:, :, 1]
canal_azul = imagen_opencv_rgb[:, :, 2]

# Calculamos promedios para tener una referencia rápida.
print(f"Promedio del canal rojo: {np.mean(canal_rojo):.2f}")
print(f"Promedio del canal verde: {np.mean(canal_verde):.2f}")
print(f"Promedio del canal azul: {np.mean(canal_azul):.2f}")


In [ ]:
# Visualizamos los canales en escala de grises y con colormaps asociados.
fig, ejes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)

ejes[0, 0].imshow(canal_rojo, cmap="gray")
ejes[0, 0].set_title("Canal rojo en grises", fontweight="bold", loc="left")
ejes[0, 0].axis("off")

ejes[0, 1].imshow(canal_verde, cmap="gray")
ejes[0, 1].set_title("Canal verde en grises", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[0, 2].imshow(canal_azul, cmap="gray")
ejes[0, 2].set_title("Canal azul en grises", fontweight="bold", loc="left")
ejes[0, 2].axis("off")

ejes[1, 0].imshow(canal_rojo, cmap="Reds")
ejes[1, 0].set_title("Canal rojo con colormap", fontweight="bold", loc="left")
ejes[1, 0].axis("off")

ejes[1, 1].imshow(canal_verde, cmap="Greens")
ejes[1, 1].set_title("Canal verde con colormap", fontweight="bold", loc="left")
ejes[1, 1].axis("off")

ejes[1, 2].imshow(canal_azul, cmap="Blues")
ejes[1, 2].set_title("Canal azul con colormap", fontweight="bold", loc="left")
ejes[1, 2].axis("off")

plt.show()


**Pregunta para el grupo:**

Si el globo naranja parece combinar rojo alto, algo de verde y poco azul, ¿qué tipo de condiciones esperarían usar? ¿Mayor que, menor que o una combinación de ambas?


## 6. Segmentación artesanal en RGB

En vez de ir directo a HSV, primero vamos a hacer una segmentación “artesanal” por canal. Esta parte está inspirada en los integradores: mirar los canales, proponer umbrales y ver qué ocurre.

Acá conviene tratar los valores como variables editables para la exposición.


In [ ]:
# Definimos umbrales iniciales para buscar naranjas.
umbral_rojo_min = 170
umbral_verde_min = 90
umbral_verde_max = 230
umbral_azul_max = 150

# Creamos una condición por canal.
mascara_rojo = canal_rojo >= umbral_rojo_min
mascara_verde = np.logical_and(canal_verde >= umbral_verde_min, canal_verde <= umbral_verde_max)
mascara_azul = canal_azul <= umbral_azul_max

# Combinamos las tres condiciones.
mascara_rgb = np.logical_and(mascara_rojo, np.logical_and(mascara_verde, mascara_azul))


In [ ]:
# Mostramos cada condición y la máscara final.
fig, ejes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

ejes[0, 0].imshow(mascara_rojo, cmap="gray")
ejes[0, 0].set_title("Condición en rojo", fontweight="bold", loc="left")
ejes[0, 0].axis("off")

ejes[0, 1].imshow(mascara_verde, cmap="gray")
ejes[0, 1].set_title("Condición en verde", fontweight="bold", loc="left")
ejes[0, 1].axis("off")

ejes[1, 0].imshow(mascara_azul, cmap="gray")
ejes[1, 0].set_title("Condición en azul", fontweight="bold", loc="left")
ejes[1, 0].axis("off")

ejes[1, 1].imshow(mascara_rgb, cmap="gray")
ejes[1, 1].set_title("Máscara RGB combinada", fontweight="bold", loc="left")
ejes[1, 1].axis("off")

plt.show()


Esta máscara no está “bien” o “mal” en abstracto. Lo interesante es mirar qué zonas aparecen, qué zonas sobran y qué zonas faltan. Ahí empieza la discusión técnica real.


## 7. Dos estrategias de limpieza

Ahora vamos a comparar dos maneras de limpiar la máscara:

1. una estrategia canónica con morfología matemática;
2. una estrategia artesanal inspirada en los integradores, eliminando filas y columnas con muy poca activación.

La comparación es útil porque muestra que una solución “casera” puede servir, pero no siempre generaliza bien.


In [ ]:
# Convertimos la máscara booleana a enteros para poder operar con OpenCV.
mascara_binaria = (mascara_rgb.astype(np.uint8) * 255)

# Estrategia 1: apertura y clausura con morfología matemática.
kernel = np.ones((5, 5), dtype=np.uint8)
mascara_morfologica = cv2.morphologyEx(mascara_binaria, cv2.MORPH_OPEN, kernel, iterations=1)
mascara_morfologica = cv2.morphologyEx(mascara_morfologica, cv2.MORPH_CLOSE, kernel, iterations=2)

# Estrategia 2: limpieza por filas y columnas con poca activación.
umbral_fila = 8
umbral_columna = 10
mascara_artesanal = mascara_binaria.copy()
alto, ancho = mascara_artesanal.shape

for fila in range(alto):
    if np.sum(mascara_artesanal[fila, :] > 0) < umbral_fila:
        mascara_artesanal[fila, :] = 0

for columna in range(ancho):
    if np.sum(mascara_artesanal[:, columna] > 0) < umbral_columna:
        mascara_artesanal[:, columna] = 0


In [ ]:
# Comparamos la máscara original con las dos limpiezas.
fig, ejes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

ejes[0].imshow(mascara_binaria, cmap="gray")
ejes[0].set_title("Máscara inicial", fontweight="bold", loc="left")
ejes[0].axis("off")

ejes[1].imshow(mascara_morfologica, cmap="gray")
ejes[1].set_title("Limpieza morfológica", fontweight="bold", loc="left")
ejes[1].axis("off")

ejes[2].imshow(mascara_artesanal, cmap="gray")
ejes[2].set_title("Limpieza por filas y columnas", fontweight="bold", loc="left")
ejes[2].axis("off")

plt.show()


**Pregunta para la discusión:**

¿Cuál de las dos limpiezas conserva mejor la forma del globo? ¿Cuál parece más específica para este caso puntual? ¿Cuál creen que generalizaría mejor si cambiáramos de imagen?


## 8. Delimitar la región y marcar su borde

Una vez que la máscara quedó más limpia, podemos usarla para construir una delimitación sencilla. Acá vamos a recuperar otra idea de los integradores: calcular un rectángulo y marcar bordes de forma manual para entender el mecanismo.


In [ ]:
# Elegimos la máscara morfológica como base para las mediciones.
mascara_trabajo = mascara_morfologica.copy()

# Buscamos los índices de los píxeles activos.
filas_activas, columnas_activas = np.where(mascara_trabajo > 0)
fila_min = int(np.min(filas_activas))
fila_max = int(np.max(filas_activas))
columna_min = int(np.min(columnas_activas))
columna_max = int(np.max(columnas_activas))

print(f"Fila mínima: {fila_min}")
print(f"Fila máxima: {fila_max}")
print(f"Columna mínima: {columna_min}")
print(f"Columna máxima: {columna_max}")


In [ ]:
# Detectamos bordes de manera manual comparando vecinos horizontales y verticales.
borde = np.zeros_like(mascara_trabajo, dtype=np.uint8)
alto, ancho = borde.shape

for fila in range(alto):
    for columna in range(1, ancho):
        if mascara_trabajo[fila, columna] != mascara_trabajo[fila, columna - 1]:
            borde[fila, columna] = 255
            borde[fila, columna - 1] = 255

for fila in range(1, alto):
    for columna in range(ancho):
        if mascara_trabajo[fila, columna] != mascara_trabajo[fila - 1, columna]:
            borde[fila, columna] = 255
            borde[fila - 1, columna] = 255

# Preparamos una versión anotada de la imagen original.
imagen_anotada = imagen_opencv_rgb.copy()
imagen_anotada[borde > 0] = [255, 255, 0]
cv2.rectangle(
    imagen_anotada,
    (columna_min, fila_min),
    (columna_max, fila_max),
    (255, 0, 0),
    3,
)

fig, ejes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
ejes[0].imshow(mascara_trabajo, cmap="gray")
ejes[0].set_title("Máscara final", fontweight="bold", loc="left")
ejes[0].axis("off")

ejes[1].imshow(borde, cmap="gray")
ejes[1].set_title("Borde detectado manualmente", fontweight="bold", loc="left")
ejes[1].axis("off")

ejes[2].imshow(imagen_anotada)
ejes[2].set_title("Borde y rectángulo sobre la imagen", fontweight="bold", loc="left")
ejes[2].axis("off")

plt.show()


Este bloque no busca competir con las funciones automáticas de OpenCV. Su valor está en otro lado: vuelve visible cómo una máscara se transforma en una región delimitada y luego en una anotación concreta sobre la imagen original.


## 9. Preguntas de cierre para la exposición

Estas preguntas pueden servir para cerrar la discusión con el grupo:

1. ¿Qué decisión les pareció más importante: el tipo de muestreo, la cuantización o la elección de umbrales?
2. ¿En qué momento la exploración dejó de ser visual y pasó a apoyarse también en métricas o tablas?
3. ¿Qué parte del procedimiento fue más sensible a pequeños cambios de parámetros?
4. ¿Qué estrategia conservarían como “método general” y cuál dejarían como solución específica para esta imagen?


## Actividad breve

Elegí un color distinto del naranja y repetí el bloque de segmentación. Después escribí una respuesta breve donde indiques:

- qué hipótesis inicial formulaste a partir de los canales;
- qué umbrales probaste primero;
- qué salió mal en la primera prueba;
- qué ajuste hiciste para mejorar la máscara;
- si preferiste la limpieza morfológica o la artesanal, y por qué.


## Cierre

Este laboratorio muestra una idea importante: procesar una imagen no es solo aplicar funciones, sino aprender a tomar decisiones con criterio. Comparar herramientas, detectar diferencias pequeñas, ajustar parámetros y justificar por qué una salida resulta más útil que otra forma parte del trabajo real con imágenes.

Justamente por eso este cuaderno está pensado como una exploración guiada: para que el ayudante no solo muestre resultados, sino también una manera de pensar técnicamente frente a un problema visual.
